# Plant Disease Classification Using Transfer Learning

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 2. Configuration

In [ ]:
DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints")

BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
IMAGE_SIZE = 224

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)


## 3. Image Preprocessing

In [ ]:
weights = ResNet18_Weights.DEFAULT
preprocess_info = weights.transforms()

mean = preprocess_info.mean
std = preprocess_info.std

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


## 4. Load the Dataset

In [ ]:
train_dataset = datasets.ImageFolder(
    DATA_DIR / "train",
    transform=train_transform,
)

val_dataset = datasets.ImageFolder(
    DATA_DIR / "val",
    transform=val_transform,
)

if train_dataset.classes != val_dataset.classes:
    raise ValueError("Training and validation classes do not match.")

class_names = train_dataset.classes
num_classes = len(class_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Classes:", class_names)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


## 5. Build the Model

In [ ]:
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

print(model.fc)


## 6. Class-Weighted Loss and Optimizer

In [ ]:
class_counts = torch.bincount(
    torch.tensor(train_dataset.targets),
    minlength=num_classes,
).float()

if (class_counts == 0).any():
    raise ValueError("Every class must have at least one training image.")

class_weights = class_counts.sum() / (num_classes * class_counts)
class_weights = class_weights.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=LEARNING_RATE,
)

print("Class counts:", class_counts.tolist())
print("Class weights:", class_weights.tolist())


## 7. Training Function

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.eval()
    model.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total


## 8. Evaluation Function

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += batch_size

            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())

    return (
        running_loss / total,
        correct / total,
        all_labels,
        all_predictions,
    )


## 9. Train the Model

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

best_val_accuracy = -1.0
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(NUM_EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )

    val_loss, val_accuracy, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_accuracy:.3f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_accuracy:.3f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "classes": class_names,
                "epoch": epoch + 1,
                "val_accuracy": val_accuracy,
            },
            CHECKPOINT_DIR / "best_model.pth",
        )

print("Best validation accuracy:", best_val_accuracy)


## 10. Plot Training History

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_accuracy"], label="Train accuracy")
plt.plot(epochs, history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


## 11. Load the Best Checkpoint

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_DIR / "best_model.pth",
    map_location=DEVICE,
    weights_only=True,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

class_names = checkpoint["classes"]

print("Best epoch:", checkpoint["epoch"])
print("Best validation accuracy:", checkpoint["val_accuracy"])
print("Classes:", class_names)


## 12. Load the Test Dataset

In [ ]:
test_dataset = datasets.ImageFolder(
    DATA_DIR / "test",
    transform=val_transform,
)

if test_dataset.classes != class_names:
    raise ValueError("Test classes do not match checkpoint classes.")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loss, test_accuracy, y_true, y_pred = evaluate(
    model, test_loader, criterion, DEVICE
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.3f}")


## 13. Classification Report

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,
    )
)


## 14. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)

display.plot(values_format="d")
plt.title("Plant Disease Classification")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 15. Predict a Single Image

In [ ]:
def predict_image(image_path, model, class_names, transform, device):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logits = model(image_tensor)
        probabilities = torch.softmax(logits, dim=1)

    predicted_index = probabilities.argmax(dim=1).item()

    return (
        class_names[predicted_index],
        probabilities[0].cpu().tolist(),
    )


## 16. Run a Prediction

In [ ]:
image_path = "example_leaf.jpg"

predicted_class, probabilities = predict_image(
    image_path,
    model,
    class_names,
    val_transform,
    DEVICE,
)

print("Predicted class:", predicted_class)

for name, probability in zip(class_names, probabilities):
    print(f"{name}: {probability:.1%}")
